# 2 · Hasard et statistiques de base — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch02_stats/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 2.1 à 2.7 | vérifier tes exercices papier ✏️ | ✏️ | ★ à ★★ | |
| A | 2.13 à 2.18 | résumer des données : tendances centrales, dispersion, histogramme, lois, 68-95-99,7 (*prochaine session de génération*) | | | |
| B | 2.19 à 2.21 | tirer au hasard : roue de la fortune, dépendance, avec ou sans remise (*prochaine session de génération*) | | | |
| C | 2.22 à 2.24 | le bootstrap et son intervalle de confiance (*prochaine session de génération*) | | | |
| D | 2.25 à 2.32 | grande dimension, covariance et corrélation, ddof, Anscombe (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Résumer des données par un centre (moyenne, médiane, mode) et une dispersion (variance, écart-type, percentiles), et savoir lequel choisir.
- Tirer au hasard de façon reproductible, dans une loi discrète, avec ou sans remise.
- Mesurer l'incertitude d'une statistique par le bootstrap.
- Calculer et interpréter covariance et corrélation, et toujours regarder les données.

**Rappel express.** $\bar{x} = \frac{1}{n}\sum_i x_i$ ; $\mathrm{Var}(x) = \frac{1}{n - \mathrm{ddof}}\sum_i (x_i - \bar{x})^2$ et $\sigma = \sqrt{\mathrm{Var}(x)}$ ; $z_i = \frac{x_i - \bar{x}}{\sigma}$ ; loi normale : 68 %, 95 %, 99,7 % des tirages à moins de 1, 2, 3 écarts-types de la moyenne ; $\mathbb{E}[X] = \sum_k x_k\,p_k$ ; $\mathrm{Cov}(x, y) = \frac{1}{n - \mathrm{ddof}}\sum_i (x_i - \bar{x})(y_i - \bar{y})$ et $r = \frac{\mathrm{Cov}(x, y)}{\sigma_x\,\sigma_y}$. En NumPy : `rng = np.random.default_rng(seed)`, `x.var(ddof=...)`, `np.percentile`, `np.cov(x, y)`, `np.corrcoef(x, y)`.

## Partie 0 · Vérifier tes exercices papier (✏️ de 2.1 à 2.7)

Fais d'abord les exercices ✏️ de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée (par exemple `42`, `0.125`, `[7, -2]` ou `"texte"`), pas l'expression Python, sinon tu ne vérifies rien. Arrondis comme l'énoncé le demande. Les réponses pas encore remplies affichent ⏳. L'exercice ∂ 2.8 se corrige avec `05_solutions.md`.

In [2]:
# The paper exercises only use the numbers of their statements (02_exercices.md)
import math

import numpy as np

SALARIES = np.array([2300, 12000, 2100, 1900, 2800, 2400, 3500, 2100, 2600])     # 2.1
CARS = np.array([176, 144, 224, 96, 160])            # 2.2: sedans, pick-ups, minivans, SUVs, estates
CAR_TYPES = ["berline", "pick-up", "monospace", "SUV", "break"]
LOADING = np.array([4, 7, 6, 3, 10])                  # 2.4
STUDY = np.array([1, 2, 3, 4, 5])                     # 2.7: hours of revision
GRADES = np.array([2, 3, 5, 4, 6])                    # 2.7: grades

**Ex 2.1 — Moyenne, médiane et mode d'une liste de salaires**

In [3]:
wb.record("2.1a", round(SALARIES.mean()), mistakes={"il y a 9 salariés : divise la somme par 9": round(SALARIES.sum() / 10), "c'est la médiane : on demande la moyenne": 2400})
wb.record("2.1b", int(np.median(SALARIES)), mistakes={"trie d'abord les salaires : la médiane est la valeur du milieu de la liste TRIÉE": 2800, "c'est la moyenne : on demande la médiane": round(SALARIES.mean())})
wb.record("2.1c", 2100, mistakes={"le mode est la valeur la plus FRÉQUENTE, pas celle du milieu": 2400})
wb.record("2.1d", int((SALARIES < SALARIES.mean()).sum()), mistakes={"on demande ceux qui gagnent MOINS que la moyenne": 1, "compare chaque salaire à la moyenne, pas à la médiane": 4})
wb.record("2.1e", round(np.where(SALARIES == 12000, 30000, SALARIES).mean()), mistakes={"la moyenne dépend de TOUS les salaires : recalcule-la avec le nouveau": round(SALARIES.mean())})
wb.record("2.1f", int(np.median(np.where(SALARIES == 12000, 30000, SALARIES))), mistakes={"c'est la nouvelle moyenne : on demande la médiane": round(np.where(SALARIES == 12000, 30000, SALARIES).mean()), "trie d'abord les salaires : la médiane est la valeur du milieu de la liste TRIÉE": 2800})

WB_ANSWER {"hash": "f4633d605a6263501548fa92e6649630adbcd5b2c449307069331b077099d4a4", "id": "2.1a", "kind": "int", "magnitude": 3, "mistakes": {"6e431e5e36849a7008b49df24efa776e4ea0d3dfaeaeb99301eb23414d4625a8": "il y a 9 salariés : divise la somme par 9", "faa9461aa9a729784742680a3c30f7e94d11e5604bce7385475bbf5420f33443": "c'est la médiane : on demande la moyenne"}, "sign": 1}
📝 Ex 2.1a : réponse enregistrée (int).
WB_ANSWER {"hash": "24c261befcc9f2721145dd61510bc6cdc1af7eedc17662434aef0fa655f24fb5", "id": "2.1b", "kind": "int", "magnitude": 3, "mistakes": {"2f4434fa662dacf80e2a0def49d804d824e669e6f7345f9565dfda97c1cb8f95": "c'est la moyenne : on demande la médiane", "4c372534b370916219cfbc6baea9f92f2c12c17f8cb8faa20d350128e27f76dc": "trie d'abord les salaires : la médiane est la valeur du milieu de la liste TRIÉE"}, "sign": 1}
📝 Ex 2.1b : réponse enregistrée (int).
WB_ANSWER {"hash": "67f4d34e0a12a4319894de36a7ad983babe2bdccf305fa25c1e94f4eff26ab99", "id": "2.1c", "kind": "int", "ma

**Ex 2.2 — De la casse de voitures à la distribution de probabilité**

In [4]:
wb.record("2.2a", CARS[2] / CARS.sum(), decimals=2, mistakes={"divise le nombre de monospaces par le nombre TOTAL de voitures": 224})
wb.record("2.2b", (CARS / CARS.sum()).tolist(), decimals=2, mistakes={"ce sont les comptages : divise chacun par le total": CARS.tolist(), "exprime les probabilités entre 0 et 1, pas en pourcentage": (100 * CARS / CARS.sum()).tolist()})
wb.record("2.2c", 360 * CARS[2] / CARS.sum(), decimals=1, mistakes={"c'est un pourcentage : un tour complet de roue fait 360°": 28, "multiplie la probabilité par 360° (un tour complet)": 0.28})
wb.record("2.2d", 1 - CARS[3] / CARS.sum(), decimals=2, mistakes={"c'est la probabilité de tirer un SUV : on demande l'événement contraire": 0.12})
wb.record("2.2e", 50 * CARS[1] / CARS.sum(), decimals=1, mistakes={"c'est la probabilité d'un seul tirage : multiplie-la par le nombre de tirages": 0.18})
wb.record("2.2f", np.cumsum(CARS / CARS.sum()).tolist(), decimals=2, mistakes={"ce sont les probabilités elles-mêmes : additionne-les au fur et à mesure": (CARS / CARS.sum()).tolist()})
wb.record("2.2g", CAR_TYPES[int(np.searchsorted(np.cumsum(CARS / CARS.sum()), 0.71, side='right'))], mistakes={"le type choisi est le premier dont la somme cumulée est strictement plus grande que u": "monospace", "compare u aux sommes cumulées de la question f, dans l'ordre de l'énoncé": "break"})

WB_ANSWER {"decimals": 2, "hash": "dd0bad7c634b7a19ca07d30b0a56a1a2e235e168e19a7a2fec68794978db238c", "hash_coarse": "f6d268734dee1540cb7ff82deed60079862555846c4b2763c980a1d71a65e73c", "id": "2.2a", "kind": "float", "magnitude": -1, "mistakes": {"91ab46b571504e452510fc1aa4889b203b4d74ac03d3cd88f9c4119f91b1086e": "divise le nombre de monospaces par le nombre TOTAL de voitures"}, "sign": 1}
📝 Ex 2.2a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "element_hashes": ["dfa90d1ca5518f95c1a134862cb24d5ca588040e0e598ab832cb33243f34bee8", "512cc2614bba0760bc0f632562151ac4433fc9acb9b0474f1addaa76ab3fa9ce", "5dc2f41f3dba1c05e5d7512f0abb0b6b766a7d7b74569f31a426622172e7c1b6", "39ca1493d32c61521400823ac3ae82af48e9f7f9f3d33348c25f937652230222", "ff802e36498d7be3851641978b629aa705357d55911968a4d64d676bba8f6e22"], "hash": "379582836b338781f0efe452da8a620e280088cc115502982dbfc410d917b401", "id": "2.2b", "kind": "array", "mistakes": {"5a149b8f01df38bd38ffcf7851d433b00f1acc0106d7a

**Ex 2.3 — La règle 68-95-99,7 sur les nageoires des manchots**

In [5]:
wb.record("2.3a", [190 - 6.5, 190 + 6.5], decimals=1, mistakes={"c'est l'intervalle à 2 écarts-types (95 %)": [177, 203], "c'est l'intervalle à 3 écarts-types (99,7 %)": [170.5, 209.5]})
wb.record("2.3b", [190 - 2 * 6.5, 190 + 2 * 6.5], decimals=1, mistakes={"c'est l'intervalle à 1 écart-type (68 %)": [183.5, 196.5], "c'est l'intervalle à 3 écarts-types (99,7 %)": [170.5, 209.5]})
wb.record("2.3c", (1 - 0.95) / 2, decimals=3, mistakes={"les 5 % hors de l'intervalle se partagent entre les DEUX côtés (symétrie)": 0.05, "c'est la part EN DESSOUS de 203 mm": 0.975, "c'est la part DANS l'intervalle": 0.95, "l'énoncé demande d'utiliser la règle 68-95-99,7, pas une table de la loi normale": 0.023})
wb.record("2.3d", round(151 * 0.025), mistakes={"seuls ceux AU-DESSUS de 203 mm comptent, pas ceux des deux côtés": round(151 * 0.05), "arrondis au plus proche, ne tronque pas (et utilise la règle 68-95-99,7, pas une table)": 3})
wb.record("2.3e", (210 - 190) / 6.5, decimals=2, mistakes={"arrondis au plus proche, ne tronque pas": 3.07, "divise l'écart à la moyenne par l'écart-type": 20})

WB_ANSWER {"decimals": 1, "element_hashes": ["c9fa6ef4ff33b0a7b8d288dcd7c40f536b7c2221fadd7523ded35512a7cd0fe4", "34ba2d59d5acfc9f283020333d3055a6b9b76feb8784b028d20475017fa216b1"], "hash": "9def0ee66695c7aff34759c1c6fde30497d7140d7c8b6d2490ef3d89a33d7153", "id": "2.3a", "kind": "array", "mistakes": {"e290f64f6ec18137d6235dd822441f84bad0e093c70bbcb654f78123c62ff9fb": "c'est l'intervalle à 2 écarts-types (95 %)", "f3e769ea92307f06005b70ef9933ffb5631e0701ee86eaed15498dffdd72071a": "c'est l'intervalle à 3 écarts-types (99,7 %)"}, "shape": [2]}
📝 Ex 2.3a : réponse enregistrée (array, 1 décimale(s)).
WB_ANSWER {"decimals": 1, "element_hashes": ["c22604505d2e02266bc3e0cfa1037eb257cfa19fa1931d7e97bd52902170af21", "d9e8662e88f8561cce31dc0d00591e59ed6dd286b6a00cc5f5107dbebc934511"], "hash": "d8a473a7e6989a5b74f39f4bb58ea87674c7b8326bc7488a5764f6b050300930", "id": "2.3b", "kind": "array", "mistakes": {"1cf43336e925c035c0325f0fb1041bf58717825e3b6d88c41432b5cde760b8f6": "c'est l'intervalle à 1 éca

**Ex 2.4 — Variance : diviser par N ou par N − 1 ?**

In [6]:
wb.record("2.4a", int(LOADING.mean()))
wb.record("2.4b", int(((LOADING - LOADING.mean()) ** 2).sum()), mistakes={"élève chaque écart au carré AVANT d'additionner (la somme des écarts vaut toujours 0)": 0, "ce sont les écarts en valeur absolue : il faut leurs CARRÉS": int(np.abs(LOADING - LOADING.mean()).sum())})
wb.record("2.4c", LOADING.var(), decimals=2, mistakes={"ddof = 0 : divise par n, pas par n − 1": LOADING.var(ddof=1), "c'est l'écart-type : on demande la variance": LOADING.std()})
wb.record("2.4d", LOADING.var(ddof=1), decimals=2, mistakes={"ddof = 1 : divise par n − 1, pas par n": LOADING.var()})
wb.record("2.4e", LOADING.std(), decimals=3, mistakes={"c'est la variance : prends sa racine carrée": LOADING.var(), "c'est l'écart-type avec ddof = 1": LOADING.std(ddof=1)})
wb.record("2.4f", LOADING.std(ddof=1), decimals=3, mistakes={"c'est la variance : prends sa racine carrée": LOADING.var(ddof=1), "c'est l'écart-type avec ddof = 0": LOADING.std()})
wb.record("2.4g", (LOADING ** 2).mean(), decimals=2, mistakes={"c'est le carré de la moyenne : on demande la moyenne des carrés": 36, "c'est la somme des carrés : divise-la par le nombre de valeurs": int((LOADING ** 2).sum())})
wb.record("2.4h", (LOADING + 100).var(), decimals=2, mistakes={"ajouter 100 décale les valeurs ET leur moyenne : les écarts ne changent pas": 106})
wb.record("2.4i", (LOADING * 1000).std(), decimals=1, mistakes={"c'est la variance : elle est multipliée par 1 000², l'écart-type par 1 000": (LOADING * 1000).var(), "multiplier les valeurs par 1 000 multiplie aussi les écarts": LOADING.std(), "tu es reparti de l'écart-type arrondi : pars de sa valeur exacte (la racine de la variance)": 2449.0})

WB_ANSWER {"hash": "95cf0084678a7d0b50f9fff8ad01c9daa38d1805f3eae14e735cc6749020307b", "id": "2.4a", "kind": "int", "magnitude": 0, "sign": 1}
📝 Ex 2.4a : réponse enregistrée (int).
WB_ANSWER {"hash": "a239ef4ea2a3219ac1f2d58be2abdf91db98d1b8bf4208df4e275f118edf15ad", "id": "2.4b", "kind": "int", "magnitude": 1, "mistakes": {"06a02b2f93e5e8de601280533f56c32d8f75216c6824a4803e4fcc0eb5684025": "ce sont les écarts en valeur absolue : il faut leurs CARRÉS", "191b6e7928e4a125efa46c8ff795037f45be7c1d8e42f1ce2b6b4e4554c5bca0": "élève chaque écart au carré AVANT d'additionner (la somme des écarts vaut toujours 0)"}, "sign": 1}
📝 Ex 2.4b : réponse enregistrée (int).
WB_ANSWER {"decimals": 2, "hash": "16f81a5b823dca6801cc5dc6d2fdfa7cd560cb9e79f14116193262a93e5b088f", "hash_coarse": "3447ab8320f44ce05f382424842765b2379522ee797607ef2a1b652318b831d6", "id": "2.4c", "kind": "float", "magnitude": 0, "mistakes": {"2570bc369cc433ad2881f791f285c93383486349c534cee2da4dfb2ec7b33bdd": "ddof = 0 : divise pa

**Ex 2.5 — Espérances : Bernoulli, multinoulli et jeu de hasard**

In [7]:
wb.record("2.5a", 0.3, decimals=2)
wb.record("2.5b", 0.3 * (1 - 0.3), decimals=2, mistakes={"c'est l'espérance : la variance d'une loi de Bernoulli vaut p(1 − p)": 0.3, "c'est p² : la variance vaut p(1 − p)": 0.09})
wb.record("2.5c", 200 * 0.3, decimals=1, mistakes={"c'est l'espérance d'un seul lancer : multiplie par le nombre de lancers": 0.3})
wb.record("2.5d", sum(range(1, 21)) / 20, decimals=1, mistakes={"la moyenne de 1, 2, …, 20 vaut (1 + 20) / 2": 10, "divise la somme des faces par le nombre de faces": sum(range(1, 21))})
wb.record("2.5e", 1 * 0.1 + 2 * 0.2 + 3 * 0.3 + 4 * 0.4, decimals=1, mistakes={"pondère chaque numéro par SA probabilité (ce n'est pas la moyenne de 1, 2, 3, 4)": 2.5})
wb.record("2.5f", [0, 0, 1, 0], mistakes={"les classes sont numérotées de 1 à 4 : la classe 3 est la troisième composante": [0, 0, 0, 1]})
wb.record("2.5g", [0.1, 0.2, 0.3, 0.4], decimals=2, mistakes={"les classes ne sortent pas toutes aussi souvent : pense à la fréquence de chacune": [0.25, 0.25, 0.25, 0.25]})
wb.record("2.5h", 20 / 20 + 5 * 3 / 20 - 2, decimals=2, mistakes={"déduis la mise de 2 € du gain moyen": 20 / 20 + 5 * 3 / 20, "la mise n'est jamais rendue : tu paies 2 € à chaque partie, même quand tu gagnes": 0.15})
wb.record("2.5i", -400 * (20 / 20 + 5 * 3 / 20 - 2), decimals=1, mistakes={"le gain de l'organisateur est l'opposé de celui du joueur": 400 * (20 / 20 + 5 * 3 / 20 - 2), "c'est le gain moyen d'une seule partie : multiplie par le nombre de parties": 0.25})

WB_ANSWER {"decimals": 2, "hash": "aa246ea00c1e407d787049b9b368e2c49f547344d881423e6891affcb3556fc0", "hash_coarse": "c29f6ff3ff85ef5e61a42abaeb1255b8ced7c3ef53bc90843ede71077fb177fd", "id": "2.5a", "kind": "float", "magnitude": -1, "sign": 1}
📝 Ex 2.5a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "e0d1bcf682560303d47e4c99193c80a718f744d295e7188d51aa5d475b4fdf8b", "hash_coarse": "3150c2be1eb6febb0917d3d64d2780df20cd4a5353988be420dc12204f3277ea", "id": "2.5b", "kind": "float", "magnitude": -1, "mistakes": {"34371fede7651ed2c7d5dd03628d2388a686a21ce61bdfb6ff9ba6a67f8e6c5a": "c'est l'espérance : la variance d'une loi de Bernoulli vaut p(1 − p)", "c554b3ee82fe6e36d40c438b78ee674c07084024a53dc42b2963c86820000ab5": "c'est p² : la variance vaut p(1 − p)"}, "sign": 1}
📝 Ex 2.5b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 1, "hash": "88823c0399dfcfb890347476ef9de724fc9d1e62110421cff90c23ecb094b833", "hash_coarse": "99ffa01785f9d4f640cd

**Ex 2.6 — Compter les tirages avec et sans remise**

In [8]:
wb.record("2.6a", math.comb(5, 2), mistakes={"sans tenir compte de l'ordre, AB et BA forment le même jeu": 20})
wb.record("2.6b", math.comb(5 + 2 - 1, 2), mistakes={"sans tenir compte de l'ordre, AB et BA ne comptent qu'une fois": 25, "avec remise, AA, BB… sont possibles aussi": 10})
wb.record("2.6c", 5 ** 2, mistakes={"ici l'ordre compte : AB et BA sont deux suites différentes": 15, "avec remise, un exemple peut sortir deux fois (AA)": 20})
wb.record("2.6d", math.comb(7, 3), mistakes={"sans tenir compte de l'ordre : chaque groupe de 3 correspond à 3! = 6 suites": 210, "sans remise, un exemple ne peut pas sortir deux fois": 343})
wb.record("2.6e", 5 ** 5, mistakes={"il y a 5 tirages, pas 2": 25, "avec remise : chacun des 5 tirages a 5 issues possibles": 120})
wb.record("2.6f", (1 - 1 / 5) ** 5, decimals=4, mistakes={"on veut l'absence à CHACUN des 5 tirages (indépendants)": 0.8, "c'est la probabilité d'apparaître AU MOINS une fois": 1 - 0.8 ** 5, "c'est la probabilité d'être tiré à un tirage donné": 0.2, "c'est la limite quand n devient très grand ; ici n = 5 : calcule la valeur exacte": math.exp(-1)})
wb.record("2.6g", (1 - 1 / 1000) ** 1000, decimals=4, mistakes={"c'est la limite quand n devient infini : calcule la valeur exacte pour n = 1 000": math.exp(-1)})
wb.record("2.6h", 1 - (1 - 1 / 1000) ** 1000, decimals=4, mistakes={"c'est la part des exemples ABSENTS du rééchantillon": (1 - 1 / 1000) ** 1000, "c'est la limite 1 − e^(−1) : calcule la valeur exacte pour n = 1 000": 1 - math.exp(-1)})

WB_ANSWER {"hash": "195ba363a054b7c8f70c349f6cd9698c1728bc06f6e69ac605ae87639dc63ae5", "id": "2.6a", "kind": "int", "magnitude": 1, "mistakes": {"e776a306e53add25800947945e89c0630ccb6075293704cb285d382478814e32": "sans tenir compte de l'ordre, AB et BA forment le même jeu"}, "sign": 1}
📝 Ex 2.6a : réponse enregistrée (int).
WB_ANSWER {"hash": "9c0b4072d128a08c7af03e48a57f5230eaa6d4cce9122db2febbfad632cfe577", "id": "2.6b", "kind": "int", "magnitude": 1, "mistakes": {"25869adaa8f4495d86f11d89434630c984186c1e66edab18676b4a428fcdda28": "avec remise, AA, BB… sont possibles aussi", "66721c913d29557d371385c167f3b133d92376308e6042b2ccd8b7db31d67d33": "sans tenir compte de l'ordre, AB et BA ne comptent qu'une fois"}, "sign": 1}
📝 Ex 2.6b : réponse enregistrée (int).
WB_ANSWER {"hash": "e946211b3b23dd31c157c34db96c391658ae51250d859d6af55b665fe03d1266", "id": "2.6c", "kind": "int", "magnitude": 1, "mistakes": {"0c12c2da48bdca774e9accbc6a7b848e5ea463536010050985b11e45946b6f8b": "ici l'ordre compt

**Ex 2.7 — Covariance et corrélation de cinq points à la main**

In [9]:
wb.record("2.7a", [STUDY.mean(), GRADES.mean()], decimals=1)
wb.record("2.7b", float(((STUDY - STUDY.mean()) * (GRADES - GRADES.mean())).sum()), decimals=1, mistakes={"additionne les PRODUITS des écarts de x et de y, pas les écarts eux-mêmes": 0, "c'est déjà la covariance (divisée par n) : on demande la somme des produits": 1.8})
wb.record("2.7c", np.cov(STUDY, GRADES, ddof=0)[0, 1], decimals=2, mistakes={"divise la somme des produits par le nombre de points": 9, "ddof = 0 : divise par n, pas par n − 1": np.cov(STUDY, GRADES, ddof=1)[0, 1]})
wb.record("2.7d", np.cov(STUDY, GRADES, ddof=1)[0, 1], decimals=2, mistakes={"ddof = 1 : divise par n − 1, pas par n": np.cov(STUDY, GRADES, ddof=0)[0, 1]})
wb.record("2.7e", [STUDY.std(), GRADES.std()], decimals=3, mistakes={"ce sont les variances : prends leur racine carrée": [STUDY.var(), GRADES.var()]})
wb.record("2.7f", np.corrcoef(STUDY, GRADES)[0, 1], decimals=3, mistakes={"tu as mélangé ddof = 1 (covariance) et ddof = 0 (écarts-types) : une corrélation ne dépasse jamais 1": np.cov(STUDY, GRADES, ddof=1)[0, 1] / (STUDY.std() * GRADES.std()), "divise la covariance par le PRODUIT des deux écarts-types": 0.45, "tu as mélangé ddof = 0 (covariance) et ddof = 1 (écarts-types) : garde le même ddof partout": np.cov(STUDY, GRADES, ddof=0)[0, 1] / (STUDY.std(ddof=1) * GRADES.std(ddof=1))})
wb.record("2.7g", np.cov(STUDY * 60, GRADES, ddof=0)[0, 1], decimals=1, mistakes={"la covariance dépend de l'unité de x : elle change": 1.8, "seul x est multiplié par 60, pas y": 1.8 * 3600})
wb.record("2.7h", np.corrcoef(STUDY * 60, GRADES)[0, 1], decimals=3, mistakes={"c'est la covariance : la corrélation divise par les écarts-types": 108})

WB_ANSWER {"decimals": 1, "element_hashes": ["2b01e4b4abc920c6d6e2fcd4cbb641ad3f7f78a9afc1aeb128c1052084ad12ac", "0ed3708c092f13e8dd57990b10642d81b2bc3cfb5c069f43d408ca0103b45767"], "hash": "686e878d2ea4fe885a54b096ceb0c939acb7c160eb7c92e1b6f7eed41d0398a3", "id": "2.7a", "kind": "array", "shape": [2]}
📝 Ex 2.7a : réponse enregistrée (array, 1 décimale(s)).
WB_ANSWER {"decimals": 1, "hash": "851e0026d280446152f543bb6ce40aba612d368beb9325fad9290df52a78dff8", "hash_coarse": "973fd1ea6263fef61bdcb685ff114f741717ab1ee2b677107f54b927a26b0848", "id": "2.7b", "kind": "float", "magnitude": 0, "mistakes": {"9d605dbed4ce238ee3dfdd373f6c8917a33c4efac78998137e0fe3dadd7c857a": "additionne les PRODUITS des écarts de x et de y, pas les écarts eux-mêmes", "a9495e7f4df27f87832cf39338e046667a181d86ecace43d4cf870250ba1188a": "c'est déjà la covariance (divisée par n) : on demande la somme des produits"}, "sign": 1}
📝 Ex 2.7b : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "2

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu choisir entre moyenne et médiane, et calculer variance et écart-type avec le bon diviseur ?
2. Sais-tu utiliser la règle 68-95-99,7 et dire quand elle ne s'applique pas ?
3. Sais-tu calculer une covariance et une corrélation, et dire ce qu'une corrélation ne prouve pas ?

**Pour aller plus loin** : *Seeing Theory* (Brown University) et le cours *Statistiques et probabilités* de Khan Academy, cités dans la fiche. La suite : le ch. 3 (probabilités conditionnelles, mesures de la qualité d'un classifieur), puis le bootstrap revient au ch. 14 (forêts aléatoires) et la matrice de covariance au ch. 12 (analyse en composantes principales).

*Les exercices de code 2.13 à 2.32 (parties A à D, ta librairie `mylearn.stats`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 2` pour obtenir le notebook complet.*